In [3]:
import matplotlib
import platform
print ("Operating system: ", platform.system())
if "Linux" in platform.system():
    %matplotlib tk
else:
    %matplotlib qt
    
#
import matplotlib.pyplot as plt
%autosave 180
%load_ext autoreload
%autoreload 2
import numpy as np

#
import scipy
import os
import time

#
import pickle 


#
from calibration.CalibrationTools import CalibrationTools, get_binary_std_map, get_rois_stardist2d, get_img_std
from drift.drift import (make_template, compute_drift_multi_frames, correct_drift, 
                         correct_drift_single_frame, template_generation, 
                         plot_mean_vs_template, make_motion_template_and_correct_data)
from utils.utils import smooth_ca_time_series, compute_dff0



Operating system:  Windows


Autosaving every 180 seconds
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [4]:
#######################################################################
########### LOAD PRE-BMI DATA (e.g. 10-15mins recording) ##############
#######################################################################
#fname = '/home/cat/code/bmi/links/Link to DON-11733/Image_001_001.raw'
#fname = '/media/cat/4TB/donato/DON-009460/22-06-21/calibration/Image_001_001.raw'
#fname = '/media/cat/4TB/donato/DON-008499/mouse2_calibration/Image_001_001.raw'
#fname = '/media/cat/4TBSSD/donato/test/data/Image_001_001.raw'
#fname = '/home/cat/data/donato/bscope_tests/8499/data/Image_001_001.raw'
#fname = '/media/cat/4TB/donato/DON-10798/22-07-26/calibration/Image_001_001.raw'
#fname = '/media/cat/4TB1/donato/DON-10795/22-07-30/calibration/Image_001_001.raw'
#fname = '/home/cat/data/donato/bscope_tests/8499/calibration/Image_001_001.raw'
#fname = '/home/cat/code/bmi/Link to 22-06-08/calibration/Image_001_001.raw'
fname = r'F:\bmi\andres\DON2\calibration\Image_001_001.raw'


# 
bmi_c = CalibrationTools(fname)

#
bmi_c.smooth_ca_time_series = smooth_ca_time_series
bmi_c.compute_dff0 = compute_dff0

#
bmi_c.subsample = 10 # for std computation downsample to every N'th frame; the more frames the better the rois;
                  #   TODO: use correlation instead?! might be much faster; it is quit fast in other implemenations


memmap :  (10000, 512, 512)


In [5]:
##################################################################
############### MOTION CORRECTION STEP ###########################
##################################################################
# 
if False:
    start = time.time()
    bmi_c = make_motion_template_and_correct_data(bmi_c)
    plot_mean_vs_template(bmi_c)
    print ("total processing time: ", time.time()-start, " sec")
else:
    print ("Skipping template makgin step: ")
    bmi_c.template = np.mean(bmi_c.data[:1000],axis=0)
    
print ("DONE...")

Skipping template makgin step: 
DONE...


In [6]:
###################################################################
################# GET STD MAP TO GET CELL FOOTPRINTS ##############
###################################################################
# 
start = time.time()

# Filter data, then make map 
if True:
    std_map = bmi_c.filter_data_make_std_map()

# # just make std, and divide by mean
# # DOESN"T SEEM TO WORK WELL in M1; 
# MIGHT TRY IN CA3 IMPLANTS
# else:
#     idx = np.arange(bmi_c.data.shape[0])[::bmi_c.subsample]
#     data_sparse = bmi_c.data[idx]
    
#     # compute std
#     std = np.std(data_sparse.astype('float32'), axis=0)
    
#     # compute mean
#     mean = np.mean(data_sparse.astype('float32'), axis=0)
    
#     #
#     std_map = (std/mean)*100
    
#     std_map = std

#
print ("total processing time: ", time.time()-start, " sec")
print ("...DONE...")


data into analysis:  (1000, 512, 512)
 gaussian filter width:  1 , order:  0
done filtering... (TO CHECK which axis are we filtering!!)
total processing time:  9.84171438217163  sec
...DONE...


In [10]:
#################################################################
########### MAKE BINARY MAP FOR CELL REGISTRATOIN ###############
#################################################################

# get binary map
vmax = 300
smin, smax = get_binary_std_map(std_map,vmax)


In [11]:
# vmax = 3000
# smin, smax = get_binary_std_map(np.mean(bmi_c.data[:1000],axis=0),vmax)


In [12]:
##################################################

bmi_c, img_std = get_img_std(smin, smax, std_map, bmi_c)


max proj values (vmin, vmax):  103.12499999999997 135.93749999999997


In [13]:
#########################################################
########### GENERATE CELL SEGMENTATION ##################
#########################################################
min_size_roi = 100  #<---- increase to exclude really small cells
max_size_roi = 800  #<----- decrease to exclude multi-cell artificats
bmi_c.roi_centres, bmi_c.footprints = get_rois_stardist2d(img_std,
                                               min_size_roi,
                                               max_size_roi)

There are 4 registered models for 'StarDist2D':

Name                  Alias(es)
────                  ─────────
'2D_versatile_fluo'   'Versatile (fluorescent nuclei)'
'2D_versatile_he'     'Versatile (H&E nuclei)'
'2D_paper_dsb2018'    'DSB 2018 (from StarDist 2D paper)'
'2D_demo'             None
None
Found model '2D_versatile_fluo' for 'StarDist2D'.
Loading network weights from 'weights_best.h5'.
Loading thresholds from 'thresholds.json'.
Using default values: prob_thresh=0.479071, nms_thresh=0.3.
1/1 [==============================] - 0s 271ms/step


looping over cells: 100%|█████████████████████████████████████████████████████████████| 95/95 [00:00<00:00, 1013.34it/s]


In [14]:
#########################################################
########### REORDER CELLS BY SNR OR F0 ##################
##############################
###########################
order_type = 'snr'  # 'snr' or 'f0'
bmi_c.compute_roi_traces_f0_and_reorder_cells(order_type)  # this function also coputes the snr / f0s of the cells


computing roi traces for SNR indexing: 100%|███████████████████████████████████████| 1000/1000 [00:01<00:00, 827.31it/s]


In [15]:
#############################################################
############## VISUALIZE ALL CELLS AND TRACES ###############
#############################################################
#
bmi_c.scale=3  
# <--- decrease to see cell traces better
bmi_c.trace_subsample = 10       # Subsample the time series to go faster;

# visualize traces
bmi_c.visualize_traces_snr_order(std_map)

# 

memmap :  (10000, 512, 512)


In [16]:
###############################################################
########### SELECT ENSEMBEL CELLS AND VISUALIZE ###############
###############################################################

# save ensemble rois
bmi_c.ensemble1 = [10,15]
bmi_c.ensemble2 = [17,12]
both = np.hstack((bmi_c.ensemble1, bmi_c.ensemble2))
print ("all cells:", both)

#
bmi_c.show_traces_ids(both)


all cells: [10 15 17 12]


### COMPUTE THE MIN AND MAXES FOR THE SELECTED ENSMEMLES

Some important points:

1. For now we are working in pixel absolute values for each cell.  

2. A better option might be to find the maximum peak of a cell during a window and then save that value and normalize all future events by that value. (note: any online BMI filtering/chagnig of data will need to account for this).


In [17]:
# ######################################################################
# ########### RECOMPUTE TRACES WITH SINGLE FRAME PRECISION #############
# ######################################################################
bmi_c.trace_subsample = 1        # Subsample the time series to go faster;

# visualize traces
#bmi_c.compute_traces2(std_map, both)
bmi_c.compute_traces_ensembles(std_map)

print ("DONE...")

100%|███████████████████████████████████████████████████████████████████████████| 10000/10000 [00:01<00:00, 8178.42it/s]


DONE...


In [20]:
#############################################
########### RUN THRSHOLD SETTER #############
#############################################
# 
bmi_c.sample_rate = 30
bmi_c.post_reward_lockout = 10   # reward lockout in seconds
                                 # TODO: in future load/save this to disk so that BMI 
                                 #   - doesn't use differetn lockout than calibration step
bmi_c.balance_ensemble_rewards_flag = False   #this makes sure that both ensembles elicit a similar number of random rewards
bmi_c.rois_smooth_window = 5     # of frames to use to smooth the realtime signal
bmi_c.smooth_diff_function_flag = True    # use a kernel window to smooth current value

#
bmi_c.reward_rate = 0.5

# find 30% reward threshold
# We have 3 options: 
#   1) rewarding  E1-E2 going above some threshold
#   2) rewarding E2-E1 going above some threhosld
#   3) rewarding both
# The challenge is mapping the ensemble states to tone/stimulus space

#gr.find_reward_thresholds_low_and_high()
bmi_c.find_reward_thresholds_high()  # this only rewards when sound passes specific level

# this option rewards both ensembles 
#normalize_peaks = True   # this flag normalizes the peaks to make sure one ensembel
#                         # doesn't completely dominate the other;
                          # TODO: make sure that this is implemented in the bmi online component also
#bmi_c.find_reward_thresholds_absolute(normalize_peaks)


#
print ("thresholds: ", bmi_c.high)

############################################## 
bmi_c.plot_rewarded_ensembles()


#
bmi_c.reward_rate_scaling_factor = 1.0

#
bmi_c.high = bmi_c.high*bmi_c.reward_rate_scaling_factor
print ("bmi_c.high: scaled by: ", bmi_c.reward_rate_scaling_factor, ", final value:  ", bmi_c.high)


COMPUTED # of roi traces:  45


100%|███████████████████████████████████████████████████████████████████████████| 9995/9995 [00:00<00:00, 477162.53it/s]

low, high:  -1.8099361656048027 2.0188403275643303
nsec recording:  333 max # of random rewards (i.e. every 30sec)  11
 @30% reward:  5
updated rewards #:  5 -0.40892890673793075 0.4561277815858553
thresholds:  0.4561277815858553


bmi_c.high: scaled by:  1.0 , final value:   0.4561277815858553


In [21]:
#############################################
########### RUN THRSHOLD SETTER #############
#############################################

# save all data to disk
# also add the tone values here as well that will be used for the experiment
bmi_c.low_freq = 2000
bmi_c.high_freq = 16000

# 
ensemble1_footprints = []
ensemble1_contours = []
for k in bmi_c.ensemble1:
    
    # get footprints
    temp = bmi_c.footprints[k]
    temp1 = temp[0]
    temp2 = temp[1]
    temp = np.vstack((temp1,temp2))
    ensemble1_footprints.append(temp.T)
    
    # get contours
    ensemble1_contours.append(bmi_c.compute_contour_map(std_map, [k]))

# get ensembel 2 footprints/contours
ensemble2_footprints = []
ensemble2_contours = []
for k in bmi_c.ensemble2:
    # get footprints
    temp = bmi_c.footprints[k]
    temp1 = temp[0]
    temp2 = temp[1]
    temp = np.vstack((temp1,temp2))
    ensemble2_footprints.append(temp.T)
    
    # get contours
    ensemble2_contours.append(bmi_c.compute_contour_map(std_map, [k]))
    
# get ensemble f0 baselines
ensemble1_f0s = []
for k in bmi_c.ensemble1:
    # get footprints
    ensemble1_f0s.append(bmi_c.roi_f0s[k])

# get ensemble f0 baselines
ensemble2_f0s = []
for k in bmi_c.ensemble2:
    # get footprints
    ensemble2_f0s.append(bmi_c.roi_f0s[k])
    
# also grab contours of cells; both contains all cell ids
# contours = bmi_c.compute_contour_map(std_map, both)
# print ("len: ", len(contours))        

# also grab contours of cells; both contains all cell ids
contours_all_cells = bmi_c.compute_contour_map(std_map, np.arange(len(bmi_c.footprints)))
contours_all_cells = np.array(contours_all_cells, dtype=object)

# save individual pixels of each cell - currently implemented in BMI
fname_out = os.path.join(os.path.split(os.path.split(fname)[0])[0],
                        'rois_pixels_and_thresholds.npz')
np.savez(fname_out,
            
            #
            ensemble1_footprints = ensemble1_footprints,
            ensemble1_contours = ensemble1_contours,
            ensemble1_f0s = ensemble1_f0s,

            # 
            ensemble2_footprints = ensemble2_footprints,
            ensemble2_contours = ensemble2_contours,
            ensemble2_f0s = ensemble2_f0s,
         
            #
            reward_rate = bmi_c.reward_rate,
            reward_rate_scaling_factor = bmi_c.reward_rate_scaling_factor,
                  
            #
            contours_all_cells = contours_all_cells,
            #cell_centres = np.int32(bmi_c.rois)[both],
            cell_ids = both,
            #all_rois = np.int32(bmi_c.rois),
            low_threshold = bmi_c.low,
            high_threshold = bmi_c.high,
            low_freq = bmi_c.low_freq,
            high_freq = bmi_c.high_freq,
            all_roi_traces_submsampled = bmi_c.roi_traces,

            #
            sample_rate = bmi_c.sample_rate,
            post_reward_lockout = bmi_c.post_reward_lockout,
            balance_ensemble_rewards_flag = bmi_c.balance_ensemble_rewards_flag,
            rois_smooth_window = bmi_c.rois_smooth_window,
            smooth_diff_function_flag = bmi_c.smooth_diff_function_flag,
            calibration_template = bmi_c.template,
            footprints = bmi_c.footprints
             
        )

# also save the entire object as a pickle
file_pi = open(os.path.join(os.path.split(fname_out)[0], "bmi_c.obj"), 'wb') 
bmi_c.data=None
pickle.dump(bmi_c, file_pi)
print ("DONE")

npyio.py (713): Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray.


DONE


In [49]:
data = np.load('/media/cat/4TBSSD/donato/DON8498/22-06-08/rois_pixels_and_thresholds.npz',
               allow_pickle=True)

ensemble1_footprints = data['ensemble1_footprints']
contours_local = data['ensemble1_contours']

FileNotFoundError: [Errno 2] No such file or directory: '/media/cat/4TBSSD/donato/DON8498/22-06-08/rois_pixels_and_thresholds.npz'

In [4]:
cell_id =1
plt.figure()
temp = bmi_c.roi_traces[cell_id]
plt.plot((temp - np.median(temp))/np.median(temp))
plt.plot((temp - bmi_c.roi_f0s[cell_id])/bmi_c.roi_f0s[cell_id])
plt.show()

NameError: name 'bmi_c' is not defined

In [6]:
plt.figure()
for c in range(len(contours_local)):
    for k in range(len(contours_local[c])-1):

        plt.plot([contours_local[c][k][0],
                            contours_local[c][k+1][0]],
                           [contours_local[c][k][1],
                            contours_local[c][k+1][1]],
                            c='blue',
                            linewidth=5)
plt.show()


In [16]:
data1 = np.load('/media/cat/4TB1/donato/DON-010800/22-08-15/data/results_regular_filter.npz',
                allow_pickle=True)
e1 = data1['ensemble_activity']
print (e1.shape)


data2 = np.load('/media/cat/4TB1/donato/DON-010800/22-08-15/data/results_5timesteps.npz',
                allow_pickle=True)
e2 = data2['ensemble_activity']
print (e2.shape)

data3 = np.load('/media/cat/4TB1/donato/DON-010800/22-08-15/data/results_no_filter.npz',
                allow_pickle=True)
e3 = data3['ensemble_activity']
print (e3.shape)

data4 = np.load('/media/cat/4TB1/donato/DON-010800/22-08-15/data/results.npz',
                allow_pickle=True)
e4 = data4['ensemble_activity']
print (e4.shape)


#############################################################
plt.figure()

t = np.arange(e1.shape[1])/30
plt.plot(t,
         e1[0]-e1[1],
         c='blue', 
         linewidth = 4,
         label='current filter')
plt.plot(t,
         e2[0]-e2[1],
         linewidth = 4,
         c='red', label = 'mean last 5 time steps')
plt.plot(t,
         e3[0]-e3[1],
         linewidth = 4,
         c='green', label = 'no filter')
plt.plot(t,
         e4[0]-e4[1],
         linewidth = 4,
         c='black', label = 'graded filter last 5 time steps')


plt.xlabel("time (sec)", fontsize=16)
plt.ylabel("Ensemble 1 - Ensemble 2", fontsize=16)
plt.xticks(fontsize=16)
plt.yticks(fontsize=16)
plt.plot([t[0],t[-1]],
         [0,0],
         '--',
        c='black')
plt.legend()
plt.show()

(2, 3000)
(2, 3000)
(2, 3000)
(2, 3000)


In [3]:
def get_octave_frequencies(low_freq,
						   high_freq,
						   octave_size=0.25):
	#
	octaves = []

	#
	octaves.append(low_freq)
	temp = low_freq
	while True:
		temp = temp * (1 + octave_size)
		if temp > high_freq:
			break
		octaves.append(temp)
	"""
	low_freq = 1000
	high_freq = 16000
	octaves = np.arange(int(low_freq/1000), 1+int(high_freq/1000))
    
	octaves = 2**(octave_size*x)
    
	#
	return np.array(1000*octaves)
	"""
	octaves = np.arange(int(low_freq/1000), 1+int(high_freq/1000))
	octaves = 2**(octave_size*x)
	return np.array(1000*octaves)

	#

In [4]:
low_freq = 2000
high_freq = 16000

octaves = get_octave_frequencies(low_freq,high_freq,octave_size=0.25)

NameError: name 'x' is not defined